In [1]:
import picsure

In [2]:
open_hpds_session = picsure.connect(
    picsure.Platform.BDC_OPEN,
    dev_mode=True
)

picsure.http /picsure/info/resources 200 381ms in=0B out=213B retry=0
picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=1 200 179ms in=25B out=635B retry=0
picsure.connect connect 0ms


You're successfully connected to BDC Open (open access).


In [3]:
facets = open_hpds_session.facets()
facets.add("dataset_id", "phs000810")
facets.add("dataset_id", "phs000007")
facets.view()

picsure.http /picsure/proxy/dictionary-api/facets 200 202ms in=25B out=117.0KB retry=0
picsure.fn session.facets 205ms


{'dataset_id': ['phs000810', 'phs000007'],
 'Consortium_Curated_Facets': [],
 'common_data_elements': [],
 'data_source': [],
 'data_type': []}

In [4]:
results = open_hpds_session.searchDictionary("age", facets=facets)
results

picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=568571 200 280ms in=597B out=1.6MB retry=0
picsure.fn session.searchDictionary 294ms


,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\phs000810\pht004715\phv00526256\AGE_IMMI\,phv00526256,AGE_IMMI,Age of immigration among participants who were...,Continuous,phs000810,[],0.0,73.0,True,None,HCHSSOL
1,\phs000007\pht000395\phv00056587\age_s1\,phv00056587,age_s1,Age = StdyDtqa - DOB,Continuous,phs000007,[],29.0,86.0,True,None,FHS
2,\phs000007\pht000397\phv00056723\age_s2\,phv00056723,age_s2,Age (age = formdate - DOB),Continuous,phs000007,[],44.0,86.0,True,None,FHS
3,\phs000007\pht003099\phv00177938\age5\,phv00177938,age5,Age at Exam 5,Continuous,phs000007,[],26.0,96.0,True,None,FHS
4,\phs000007\pht003099\phv00177948\age10\,phv00177948,age10,Age at Exam 10,Continuous,phs000007,[],46.0,102.0,True,None,FHS
...,...,...,...,...,...,...,...,...,...,...,...,...
2037,\phs000007\pht000676\phv00066829\phrm_gp1\,phv00066829,phrm_gp1,Description/Label for pharmacological subgroup...,Categorical,phs000007,"[ACE INHIBITORS, PLAIN, ADRENERGICS FOR SYSTEM...",NaN,NaN,False,None,FHS
2038,\phs000007\pht000676\phv00066827\system1\,phv00066827,system1,Description/Label for anatomical main group - ...,Categorical,phs000007,"[ALIMENTARY TRACT AND METABOLISM, ANTIINFECTIV...",NaN,NaN,False,None,FHS
2039,\phs000007\pht000676\phv00066823\phrm_gp2\,phv00066823,phrm_gp2,Description/Label for pharmacological subgroup...,Categorical,phs000007,"[ACE INHIBITORS, PLAIN, ANTACIDS, ANTIGLAUCOMA...",NaN,NaN,False,None,FHS
2040,\phs000007\pht000676\phv00066833\MEDNAME\,phv00066833,MEDNAME,Medication name,Categorical,phs000007,"[(ANTIOXIDANT), (ETHIN. ESTRADIO, (ETHINYL EST...",NaN,NaN,False,None,FHS


In [5]:
# phs000007 - Will be used to replace the phs000810 clause
age5_phs000007 = results[results["display"] == "age5"]
age5_phs000007 = age5_phs000007[age5_phs000007["name"] == "phv00177938"]
age5_phs000007_clause = picsure.buildClause(
    age5_phs000007["conceptPath"].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    min=30,
    max=40
)

# phs000810 - Will be used to replace the phs000007 clause
age_immi_phs000810 = results[results["display"] == "AGE_IMMI"]
age_immi_phs000810_clause = picsure.buildClause(
    age_immi_phs000810["conceptPath"].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    min=30,
    max=40
)

In [6]:
fhs_facet = open_hpds_session.facets()
fhs_facet.add("dataset_id", "phs000007")
fhs_sex_results = open_hpds_session.searchDictionary("phv00253990", facets=fhs_facet)
fhs_sex_results

picsure.http /picsure/proxy/dictionary-api/facets 200 157ms in=25B out=117.0KB retry=0
picsure.fn session.facets 159ms
picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=568571 200 154ms in=297B out=627B retry=0
picsure.fn session.searchDictionary 155ms


,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\phs000007\pht004374\phv00253990\sex\,phv00253990,sex,Sex of the participant,Categorical,phs000007,"[Female, Male]",None,None,True,None,FHS


In [7]:
fhs_sex_male_clause = picsure.buildClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Male"]
)

fhsMaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_male_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_sex_female_clause = picsure.buildClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Female"]
)

fhsFemaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_female_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_FemaleAges30to40_OR_MaleAges30to40 = picsure.buildClauseGroup(
    [fhsFemaleAnd30To40, fhsMaleAnd30To40],
    operator=picsure.GroupOperator.OR
)

fhs_FemaleAges30to40_OR_MaleAges30to40.to_query_json()

{'operator': 'OR',
 'phenotypicClauses': [{'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Female']},
    {'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht003099\\phv00177938\\age5\\',
     'not': False,
     'min': 30,
     'max': 40}],
   'not': False},
  {'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Male']},
    {'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht003099\\phv00177938\\age5\\',
     'not': False,
     'min': 30,
     'max': 40}],
   'not': False}],
 'not': False}

In [8]:
open_hpds_session.runQuery(picsure.buildQuery(phenotypicFilter=fhs_FemaleAges30to40_OR_MaleAges30to40))
# Results are within expected margin. Verified results with UI. UI Result: 77±3

picsure.http /picsure/query/sync 200 193ms in=826B out=6B retry=0
picsure.fn session.runQuery 195ms


CountResult(value=78, margin=3, cap=None, raw='78 ±3')

In [9]:
# We will now replace all of the age5_phs000007_clause with age_immi_phs000810_clause
fhs_FemaleAges30to40_OR_MaleAges30to40 = picsure.replaceClause(
    fhs_FemaleAges30to40_OR_MaleAges30to40,
    age5_phs000007_clause,
    age_immi_phs000810_clause
)

fhs_FemaleAges30to40_OR_MaleAges30to40.to_query_json()

{'operator': 'OR',
 'phenotypicClauses': [{'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Female']},
    {'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000810\\pht004715\\phv00526256\\AGE_IMMI\\',
     'not': False,
     'min': 30,
     'max': 40}],
   'not': False},
  {'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Male']},
    {'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000810\\pht004715\\phv00526256\\AGE_IMMI\\',
     'not': False,
     'min': 30,
     'max': 40}],
   'not': False}],
 'not': False}

In [10]:
open_hpds_session.runQuery(picsure.buildQuery(phenotypicFilter=fhs_FemaleAges30to40_OR_MaleAges30to40))

picsure.http /picsure/query/sync 200 195ms in=834B out=4B retry=0
picsure.fn session.runQuery 196ms


CountResult(value=None, margin=None, cap=10, raw='< 10')

In [11]:
# We can also remove sub-queries from a query
fhs_Female_OR_Male = picsure.removeSubQuery(
    fhs_FemaleAges30to40_OR_MaleAges30to40,
    age_immi_phs000810_clause
)

fhs_Female_OR_Male.to_query_json()

{'operator': 'OR',
 'phenotypicClauses': [{'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Female']}],
   'not': False},
  {'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Male']}],
   'not': False}],
 'not': False}

In [12]:
open_hpds_session.runQuery(picsure.buildQuery(phenotypicFilter=fhs_Female_OR_Male))

picsure.http /picsure/query/sync 200 192ms in=578B out=8B retry=0
picsure.fn session.runQuery 193ms


CountResult(value=1273, margin=3, cap=None, raw='1273 ±3')